# 04. Time Split and Sampling Strategy

## Objective

This notebook defines the final development, validation and out-of-time testing samples for the credit risk model.

Key decisions covered here:

- Use of a time-based split instead of random split
- Exclusion of insufficiently seasoned vintages identified in Notebook 02
- Definition of Train / Validation / OOT samples
- Assessment of target distribution and sample representativeness
- Assessment of class imbalance
- Export of split datasets and summary artifacts

This notebook follows the target and modeling population decisions made in:

`02_target_definition_and_leakage.ipynb`


In [ ]:
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Project paths

The notebook expects to be executed from the `notebooks/` folder.

Input from Notebook 02:

`../data/processed/02.target_definition_and_leakage/modeling_dataset.pkl`

Outputs from this notebook are saved to:

`../data/processed/04.time_split_and_sampling_strategy/`


In [ ]:
PROJECT_ROOT = Path("..")

INPUT_DIR = PROJECT_ROOT / "data" / "processed" / "02.target_definition_and_leakage"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "04.time_split_and_sampling_strategy"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = INPUT_DIR / "modeling_dataset.pkl"

FALLBACK_INPUT_FILES = [
    INPUT_DIR / "initial_modeling_dataset.pkl",
    INPUT_DIR / "modeling_population.pkl",
    INPUT_DIR / "target_definition_dataset.pkl",
]

print(f"Input directory: {INPUT_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

## 2. Load modeling dataset from Notebook 02

The dataset should already include:

- the final binary target column `target_bad`
- the resolved loan population
- exclusion of major leakage variables
- exclusion of insufficiently seasoned vintages, if applied in Notebook 02


In [ ]:
# Load input dataset

if INPUT_FILE.exists():
    df = pd.read_pickle(INPUT_FILE)
    used_input_file = INPUT_FILE
else:
    available_fallbacks = [p for p in FALLBACK_INPUT_FILES if p.exists()]
    if available_fallbacks:
        used_input_file = available_fallbacks[0]
        df = pd.read_pickle(used_input_file)
    else:
        raise FileNotFoundError(
            "Could not find modeling dataset from Notebook 02. "
            f"Expected: {INPUT_FILE}. "
            f"Also checked: {FALLBACK_INPUT_FILES}"
        )

print(f"Loaded file: {used_input_file}")
print(f"Dataset shape: {df.shape}")
df.head()

## 3. Target and date checks

For this project we use:

- `target_bad = 1` for Charged Off / bad
- `target_bad = 0` for Fully Paid / good

`issue_d` is interpreted as the loan origination / disbursement date and is used for vintage-based sample splitting.


In [ ]:
TARGET_COL = "target_bad"
DATE_COL = "issue_d"

if TARGET_COL not in df.columns:
    raise ValueError(f"Expected target column '{TARGET_COL}' was not found. Check Notebook 02 output.")

if DATE_COL not in df.columns:
    raise ValueError(f"Expected date column '{DATE_COL}' was not found. Cannot perform time split.")

# Convert issue_d to datetime
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

if df[DATE_COL].isna().any():
    missing_issue_dates = df[DATE_COL].isna().sum()
    print(f"Warning: {missing_issue_dates:,} rows have missing/unparseable issue_d.")

# Create vintage fields
df["issue_year"] = df[DATE_COL].dt.year
df["issue_quarter"] = df[DATE_COL].dt.to_period("Q").astype(str)
df["issue_month"] = df[DATE_COL].dt.to_period("M").astype(str)

print("Target distribution:")
display(df[TARGET_COL].value_counts(dropna=False).to_frame("count"))

print("Issue date range:")
print(df[DATE_COL].min(), "to", df[DATE_COL].max())

## 4. Vintage analysis

Review of the volume and observed bad rate by issue quarter.


In [ ]:
def summarize_target(data, group_cols=None, target_col=TARGET_COL):
    if group_cols is None:
        group_cols = []

    if isinstance(group_cols, str):
        group_cols = [group_cols]

    if group_cols:
        out = (
            data.groupby(group_cols, dropna=False)
                .agg(
                    total_accounts=(target_col, "size"),
                    good_count=(target_col, lambda x: (x == 0).sum()),
                    bad_count=(target_col, lambda x: (x == 1).sum()),
                    observed_bad_rate=(target_col, "mean"),
                )
                .reset_index()
        )
    else:
        out = pd.DataFrame({
            "total_accounts": [len(data)],
            "good_count": [(data[target_col] == 0).sum()],
            "bad_count": [(data[target_col] == 1).sum()],
            "observed_bad_rate": [data[target_col].mean()],
        })

    return out

vintage_summary = summarize_target(df, "issue_quarter")
vintage_summary

In [ ]:
plot_data = vintage_summary.copy()

fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.bar(
    plot_data["issue_quarter"],
    plot_data["total_accounts"],
    alpha=0.75,
)
ax1.set_ylabel("Total accounts")
ax1.set_xlabel("Issue quarter")
ax1.tick_params(axis="x", rotation=60)
ax1.grid(axis="y", linestyle="--", alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(
    plot_data["issue_quarter"],
    plot_data["observed_bad_rate"],
    color="red",
    marker="o",
    linewidth=2.5,
)
ax2.set_ylabel("Observed bad rate", color="red")
ax2.tick_params(axis="y", labelcolor="red")

plt.title("Vintage Volume and Observed Bad Rate")
plt.tight_layout()
plt.show()

## 5. Split strategy

A random split would mix old and recent vintages and would overstate expected real-world performance.

Therefore, a temporal split is used to define the Train, Validation and Out-of-Time (OOT) samples.

| Sample | Vintage period | Purpose |
|---|---|---|
| Train | 2012Q3–2016Q4 | model development |
| Validation | 2017Q1–2017Q4 | model selection and tuning |
| OOT Test | 2018Q1–2018Q2 | out-of-time performance check |

The latest vintages, `2018Q3` and `2018Q4`, are excluded due to insufficient seasoning / maturity bias.


In [ ]:
# Define split periods

TRAIN_START = pd.Period("2012Q3", freq="Q")
TRAIN_END = pd.Period("2016Q4", freq="Q")

VALIDATION_START = pd.Period("2017Q1", freq="Q")
VALIDATION_END = pd.Period("2017Q4", freq="Q")

OOT_START = pd.Period("2018Q1", freq="Q")
OOT_END = pd.Period("2018Q2", freq="Q")

EXCLUDED_VINTAGES = ["2018Q3", "2018Q4"]

df["issue_quarter_period"] = df[DATE_COL].dt.to_period("Q")

conditions = [
    df["issue_quarter_period"].between(TRAIN_START, TRAIN_END),
    df["issue_quarter_period"].between(VALIDATION_START, VALIDATION_END),
    df["issue_quarter_period"].between(OOT_START, OOT_END),
]

choices = ["train", "validation", "oot_test"]

df["sample"] = np.select(conditions, choices, default="excluded_or_outside_scope")

split_summary = summarize_target(df, "sample").sort_values("sample")
split_summary

In [ ]:
# Detailed vintage and sample mapping

vintage_split_summary = summarize_target(df, ["sample", "issue_quarter"]).sort_values(
    ["sample", "issue_quarter"]
)

vintage_split_summary

## 6. Create final split datasets

Only `train`, `validation` and `oot_test` are retained for modelling.

Rows outside these periods remain documented in the split summary but are not used in model development.


In [ ]:
train = df[df["sample"] == "train"].copy()
validation = df[df["sample"] == "validation"].copy()
oot_test = df[df["sample"] == "oot_test"].copy()
excluded = df[df["sample"] == "excluded_or_outside_scope"].copy()

print(f"Train shape:      {train.shape}")
print(f"Validation shape: {validation.shape}")
print(f"OOT test shape:   {oot_test.shape}")
print(f"Excluded shape:   {excluded.shape}")

In [ ]:
final_split_summary = pd.concat(
    [
        summarize_target(train).assign(sample="train"),
        summarize_target(validation).assign(sample="validation"),
        summarize_target(oot_test).assign(sample="oot_test"),
        summarize_target(excluded).assign(sample="excluded_or_outside_scope"),
    ],
    ignore_index=True,
)

final_split_summary = final_split_summary[
    ["sample", "total_accounts", "good_count", "bad_count", "observed_bad_rate"]
]

final_split_summary

## 7. Representativeness checks

Comparison of key categorical distributions across samples.

In [ ]:
# Candidate categorical variables for distribution checks
candidate_categorical_vars = [
    "grade",
    "sub_grade",
    "term",
    "home_ownership",
    "verification_status",
    "purpose",
    "application_type",
    "initial_list_status",
]

categorical_vars = [c for c in candidate_categorical_vars if c in df.columns]

print("Categorical variables available for representativeness checks:")
categorical_vars

In [ ]:
def distribution_by_sample(data, variable, sample_col="sample"):
    out = (
        data[data[sample_col].isin(["train", "validation", "oot_test"])]
        .groupby([sample_col, variable], dropna=False)
        .size()
        .reset_index(name="count")
    )

    out["sample_total"] = out.groupby(sample_col)["count"].transform("sum")
    out["share"] = out["count"] / out["sample_total"]

    return out.sort_values([variable, sample_col])

categorical_distribution_tables = {}

for var in categorical_vars:
    categorical_distribution_tables[var] = distribution_by_sample(df, var)

# Show first available distribution
if categorical_vars:
    display(categorical_distribution_tables[categorical_vars[0]].head(30))

In [ ]:
# Plot selected categorical distributions by sample

for var in categorical_vars:
    pivot = (
        categorical_distribution_tables[var]
        .pivot_table(index=var, columns="sample", values="share", fill_value=0)
    )

    # Limit very high-cardinality variables if needed
    if pivot.shape[0] > 25:
        top_levels = (
            df[df["sample"] == "train"][var]
            .value_counts(dropna=False)
            .head(25)
            .index
        )
        pivot = pivot.loc[pivot.index.isin(top_levels)]

    ax = pivot.plot(kind="bar", figsize=(12, 5))
    ax.set_title(f"{var}: Distribution by Sample")
    ax.set_ylabel("Share")
    ax.set_xlabel(var)
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()

## 8. Numeric distribution checks


In [ ]:
candidate_numeric_vars = [
    "loan_amnt",
    "int_rate",
    "installment",
    "annual_inc",
    "dti",
    "fico_range_low",
    "fico_range_high",
    "revol_util",
    "revol_bal",
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "total_acc",
    "mort_acc",
    "pub_rec",
    "pub_rec_bankruptcies",
]

numeric_vars = [c for c in candidate_numeric_vars if c in df.columns]

print("Numeric variables available for distribution checks:")
numeric_vars

In [ ]:
def numeric_summary_by_sample(data, variables, sample_col="sample"):
    rows = []
    use_data = data[data[sample_col].isin(["train", "validation", "oot_test"])].copy()

    for sample_name, sample_df in use_data.groupby(sample_col):
        for var in variables:
            s = sample_df[var]
            rows.append({
                "sample": sample_name,
                "variable": var,
                "count": s.notna().sum(),
                "missing_pct": s.isna().mean(),
                "mean": s.mean(),
                "std": s.std(),
                "p1": s.quantile(0.01),
                "p5": s.quantile(0.05),
                "p25": s.quantile(0.25),
                "p50": s.quantile(0.50),
                "p75": s.quantile(0.75),
                "p95": s.quantile(0.95),
                "p99": s.quantile(0.99),
                "min": s.min(),
                "max": s.max(),
            })

    return pd.DataFrame(rows)

numeric_distribution_summary = numeric_summary_by_sample(df, numeric_vars)
numeric_distribution_summary.head(30)

## 9. Class imbalance assessment

The observed bad rate is around 20%, which is not a severe imbalance problem.

Therefore, no oversampling, undersampling or SMOTE is applied at this stage.

This is especially important for credit risk scorecards because artificial resampling can distort observed default rates, calibration and score-to-PD interpretation.


In [ ]:
class_imbalance_summary = final_split_summary.copy()
class_imbalance_summary["bad_to_good_ratio"] = (
    class_imbalance_summary["bad_count"] / class_imbalance_summary["good_count"]
)

class_imbalance_summary

## 10. Save outputs

The notebook saves:

### Datasets

- `train.pkl`
- `validation.pkl`
- `oot_test.pkl`
- `excluded_or_outside_scope.pkl`

### Artifacts

- `time_split_and_sampling_artifacts.pkl`
- `time_split_and_sampling_report.xlsx`

The Excel report contains all key split, vintage, class balance and representativeness tables.


In [ ]:
# Save split datasets
train.to_pickle(OUTPUT_DIR / "train.pkl")
validation.to_pickle(OUTPUT_DIR / "validation.pkl")
oot_test.to_pickle(OUTPUT_DIR / "oot_test.pkl")
excluded.to_pickle(OUTPUT_DIR / "excluded_or_outside_scope.pkl")

# Prepare artifacts dictionary
artifacts = {
    "used_input_file": str(used_input_file),
    "target_col": TARGET_COL,
    "date_col": DATE_COL,
    "train_period": {"start": str(TRAIN_START), "end": str(TRAIN_END)},
    "validation_period": {"start": str(VALIDATION_START), "end": str(VALIDATION_END)},
    "oot_period": {"start": str(OOT_START), "end": str(OOT_END)},
    "excluded_vintages": EXCLUDED_VINTAGES,
    "split_summary": split_summary,
    "final_split_summary": final_split_summary,
    "vintage_summary": vintage_summary,
    "vintage_split_summary": vintage_split_summary,
    "class_imbalance_summary": class_imbalance_summary,
    "categorical_distribution_tables": categorical_distribution_tables,
    "numeric_distribution_summary": numeric_distribution_summary,
}

with open(OUTPUT_DIR / "time_split_and_sampling_artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)

# Save Excel report
excel_path = OUTPUT_DIR / "time_split_and_sampling_report.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    final_split_summary.to_excel(writer, sheet_name="split_summary", index=False)
    vintage_summary.to_excel(writer, sheet_name="vintage_summary", index=False)
    vintage_split_summary.to_excel(writer, sheet_name="vintage_split", index=False)
    class_imbalance_summary.to_excel(writer, sheet_name="class_imbalance", index=False)
    numeric_distribution_summary.to_excel(writer, sheet_name="numeric_summary", index=False)

    for var, table in categorical_distribution_tables.items():
        sheet_name = f"cat_{var}"[:31]
        table.to_excel(writer, sheet_name=sheet_name, index=False)

print("Saved outputs:")
print(f"- {OUTPUT_DIR / 'train.pkl'}")
print(f"- {OUTPUT_DIR / 'validation.pkl'}")
print(f"- {OUTPUT_DIR / 'oot_test.pkl'}")
print(f"- {OUTPUT_DIR / 'excluded_or_outside_scope.pkl'}")
print(f"- {OUTPUT_DIR / 'time_split_and_sampling_artifacts.pkl'}")
print(f"- {excel_path}")

## 11. Conclusions

The final modelling strategy is:

- Use a **time-based split**, not random splitting
- Use `2012Q3–2016Q4` as the training sample
- Use `2017Q1–2017Q4` as the validation sample
- Use `2018Q1–2018Q2` as the OOT test sample
- Exclude latest vintages with insufficient seasoning / maturity bias
- SMOTE, undersampling or oversampling will not be performed at this stage
- Preserve observed class distribution for proper credit risk interpretation and later score-to-PD calibration

